# 02c. Capture fidelity: veth MTU, offloads and capture tail

Changes to `harness/` after VM run 20261022_222111, before any collection. The benign scope of notebook 02 is
unchanged.

**Design decisions**
- *veth MTU.* Each run's veth pair was created at the default MTU 1500, while the VM's uplink (ens4 on Google Cloud) has
  MTU 1460. Runs sent full-size segments (1,428-byte payload, 1,480-byte packets) that the uplink dropped, followed by
  ICMP "fragmentation needed" and resends at the start of every connection. V8's resent bytes were the same in two runs
  four days apart: 12,852 = 9 x 1,428 (Claude Code through the gateway), 11,424 = 8 x 1,428 (Codex through the
  gateway), 1,507 = 1,428 + 79 and 1,547 = 1,428 + 119 (direct runs). `netns.py` now reads the MTU of the interface
  that carries the IPv4 default route (or the default route's own mtu attribute when that is smaller), sets both veth
  ends to it, checks both ends after creation and records it in the run metadata.
- *Offloads.* With segmentation offloads on, the run's TCP stack hands the veth blocks of up to 64 KB, cut into
  wire-size segments only later, and tcpdump on the run interface records the blocks. The resent figure 12,852 matches
  one such 9-segment block. TSO, GSO and GRO are now off on both veth ends (ethtool), so the run's own segments, and
  packets merged on arrival at the VM (GRO on the uplink), are wire-size when they reach the run interface; merged
  packets keep their wire sizes but lose the microsecond gaps between their arrivals.
- *Capture tail.* In tcpdump's default buffered mode, packets still waiting in its buffer when it is stopped are never
  written. Off the VM, captures stopped 0.5 s after a 1 MB transfer wrote 1,616 of 1,737, 1,713 of 1,870 and 1,528 of
  1,648 packets that the filter received, with 0 dropped. The gate's runs wait 1 s before stopping, and in the same
  simulation their buffered captures were complete, so this is a risk for any run stopped sooner rather than a fault
  seen in the gate's captures. tcpdump now runs in immediate mode (every packet written in the same tests), and each
  capture's closing counts are recorded and checked.
- *Checks.* V2 (both runs) and V8 (all four agent runs) fail a run when: a veth end's MTU is not the uplink MTU or its
  offloads are on; the capture holds an ICMP "fragmentation needed" message; a captured packet is larger than the MTU;
  upload bytes resent exceed one full-size segment (MTU minus 40 bytes; expected 0 on a clean link); or tcpdump did not
  write every packet its filter received. Download bytes resent by the endpoint are reported, not gated. Every run
  directory gets `net.json` (uplink, MTU, offload states, capture counts). V0 reports the uplink and its MTU and needs
  ethtool, which `vm_setup.sh` now installs.
- *Reproduction off the VM.* Uplink MTU 1460, endpoint advertising MSS 1440, 1 MB upload: with the 02b `netns.py` the
  capture held 2 ICMP "fragmentation needed" messages (next-hop MTU 1460) and 12,852 resent bytes, the kernel counted
  11 resent segments, and the largest captured packet was 65,212 bytes; with this `netns.py`, 0, 0, 0 and 1,460 bytes,
  and tcpdump wrote every packet it received. In the full gate on the same simulated link, with the MTU or the offload
  setting undone, V2 and V8 failed on the matching checks; with all three changes undone (the 02b state), the direct
  Claude Code run resent 1,507 bytes (1,428 + 79, as on the VM) and each gateway run 17,136 bytes (12 x 1,428).

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/UNEXPLAINED_TRAFFIC_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'unexplained-traffic')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('harness', 'reports/harness', 'data/harness'): os.makedirs(d_, exist_ok=True)
print('Session ready. CPUs:', os.cpu_count())

## Stage A. Harness changes

In [ ]:
%%writefile harness/netns.py
"""One network namespace per run, with NAT to the internet, its own DNS settings and a packet capture on the run's own
interface, so concurrent runs never mix. Linux only, run as root.

Run k gets namespace ut<k>, a veth pair uth<k> (VM side) and utn<k> (run side) and the subnet 10.200.<k>.0/30.
Containment rules, tagged with the comment ut-run:ut<k> and removed with the run:
  - the run reaches the public internet only: private, shared and link-local ranges (which include the cloud
    metadata service) are dropped, and so is any other run's subnet;
  - the run cannot reach services on the VM itself;
  - IPv6 is off inside the run, so every capture is IPv4;
  - DNS goes to public resolvers, because the VM's own resolver is a local stub the run cannot reach.

Capture fidelity, so that the run capture holds the packets that cross the wire:
  - MTU. Both ends of the veth pair get the MTU of the VM's uplink (the interface that carries the IPv4 default route,
    or the default route's own mtu attribute when that is smaller). A veth pair left at the default 1500 on an uplink
    with a smaller MTU (1460 on Google Cloud) makes every run send full-size segments that the uplink drops, followed
    by ICMP "fragmentation needed" and resends at the start of every connection.
  - Offloads. Segmentation and receive offloads (TSO, GSO, GRO) are off on both ends. With them on, the run's TCP
    stack hands the veth blocks of up to 64 KB that are cut into wire-size segments only later, and a capture on the
    veth records the blocks, not the packets. With them off, the run's own segments, and packets merged on arrival at
    the VM (GRO on the uplink), are wire-size when they reach the run interface; merged packets keep their wire sizes
    but lose the microsecond gaps between their arrivals.
  - Capture tail. tcpdump runs in immediate mode, so every packet is written as it arrives; in its default buffered
    mode the packets waiting in its buffer when it is stopped are lost.
The MTU and offload settings are checked on both ends after creation; they and each capture's packet counts are
recorded in the run metadata (metadata()).
"""
from __future__ import annotations

import json
import os
import re
import shlex
import shutil
import signal
import subprocess
import threading

TAG = "ut-run"
OFFLOADS = {"tso": "tcp-segmentation-offload", "gso": "generic-segmentation-offload",
            "gro": "generic-receive-offload"}
RESOLVERS = ["1.1.1.1", "8.8.8.8"]
BLOCKED = ["10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16", "100.64.0.0/10", "169.254.0.0/16"]


def sh(args, check=True):
    r = subprocess.run(args, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(args)} failed: {r.stderr.strip() or r.stdout.strip()}")
    return r


def default_route() -> tuple[str, int | None]:
    """Interface of the IPv4 default route, and the route's own mtu attribute when it has one."""
    return parse_default_route(sh(["ip", "-4", "route", "show", "default"]).stdout)


def parse_default_route(text: str) -> tuple[str, int | None]:
    out = text.split()
    if "dev" not in out:
        raise RuntimeError("no IPv4 default route")
    route_mtu = None
    for i, word in enumerate(out[:-1]):
        if word == "mtu":
            value = out[i + 2] if out[i + 1] == "lock" and i + 2 < len(out) else out[i + 1]
            route_mtu = int(value) if value.isdigit() else None
            break
    return out[out.index("dev") + 1], route_mtu


def default_interface() -> str:
    return default_route()[0]


def parse_link_mtu(text: str) -> int:
    """MTU from the output of 'ip -o link show dev <name>'."""
    m = re.search(r"\bmtu (\d+)\b", text)
    if not m:
        raise RuntimeError(f"no mtu in: {text.strip()[:200]}")
    return int(m.group(1))


def link_mtu(dev: str, ns: str | None = None) -> int:
    args = ["ip"] + (["-n", ns] if ns else []) + ["-o", "link", "show", "dev", dev]
    return parse_link_mtu(sh(args).stdout)


def parse_offloads(text: str) -> dict:
    """States of the offloads in OFFLOADS from the output of 'ethtool -k <name>' ("off", "on", or "off [fixed]")."""
    states = {}
    for line in text.splitlines():
        name, sep, value = line.strip().partition(":")
        if sep and name in OFFLOADS.values():
            states[name] = value.strip()
    return states


def link_offloads(dev: str, ns: str | None = None) -> dict:
    args = (["ip", "netns", "exec", ns] if ns else []) + ["ethtool", "-k", dev]
    return parse_offloads(sh(args).stdout)


def offloads_off(states: dict) -> bool:
    return set(states) == set(OFFLOADS.values()) and all(v.split()[0] == "off" for v in states.values())


def uplink_mtu() -> dict:
    """The uplink, its MTU, the default route's mtu attribute, and the MTU the run interfaces get."""
    dev, route_mtu = default_route()
    dev_mtu = link_mtu(dev)
    return dict(uplink=dev, uplink_mtu=dev_mtu, route_mtu=route_mtu,
                mtu=min(dev_mtu, route_mtu) if route_mtu else dev_mtu)


def remove_tagged_rules(tag: str) -> int:
    """Delete every iptables rule whose comment equals tag (or starts with it when tag ends with ':')."""
    removed = 0
    for table in ("filter", "nat"):
        for line in sh(["iptables", "-w", "-t", table, "-S"]).stdout.splitlines():
            if not line.startswith("-A ") or "--comment" not in line:
                continue
            args = shlex.split(line)
            comment = args[args.index("--comment") + 1]
            if comment == tag or (tag.endswith(":") and comment.startswith(tag)):
                sh(["iptables", "-w", "-t", table, "-D"] + args[1:], check=False)
                removed += 1
    return removed


def cleanup_all() -> None:
    """Remove every run namespace, interface and rule left by earlier runs."""
    remove_tagged_rules(f"{TAG}:")
    for line in sh(["ip", "netns", "list"], check=False).stdout.splitlines():
        name = line.split()[0] if line.split() else ""
        if name.startswith("ut") and name[2:].isdigit():
            sh(["ip", "netns", "del", name], check=False)
            shutil.rmtree(f"/etc/netns/{name}", ignore_errors=True)


def parse_capture_stats(text: str) -> dict:
    """tcpdump's closing counts: packets captured (written), received by filter, dropped by kernel or interface.
    complete is true when every packet the filter received was written and none was dropped."""
    counts = {}
    for key, pattern in (("captured", r"(\d+) packets? captured"), ("received", r"(\d+) packets? received by filter"),
                         ("dropped_kernel", r"(\d+) packets? dropped by kernel"),
                         ("dropped_interface", r"(\d+) packets? dropped by interface")):
        m = re.search(pattern, text)
        counts[key] = int(m.group(1)) if m else None
    counts["complete"] = (counts["captured"] is not None and counts["captured"] == counts["received"]
                          and not counts["dropped_kernel"] and not counts["dropped_interface"])
    return counts


class Capture:
    def __init__(self, proc, path):
        self.proc, self.path = proc, path
        self.lines = []
        self.ready = threading.Event()
        self.reader = threading.Thread(target=self._read, daemon=True)
        self.reader.start()

    def _read(self):
        for line in self.proc.stderr:
            self.lines.append(line.rstrip())
            if "listening on" in line:
                self.ready.set()
        self.ready.set()

    def stop(self) -> str:
        if self.proc.poll() is None:
            self.proc.send_signal(signal.SIGINT)
            try:
                self.proc.wait(15)
            except subprocess.TimeoutExpired:
                self.proc.kill()
                self.proc.wait(5)
        self.reader.join(5)
        return "\n".join(self.lines)

    def stats(self) -> dict:
        return dict(path=self.path, **parse_capture_stats("\n".join(self.lines)))


class RunNet:
    def __init__(self, index: int):
        if not 1 <= index <= 254:
            raise ValueError("run index must be 1 to 254")
        self.index = index
        self.ns = f"ut{index}"
        self.host_if, self.ns_if = f"uth{index}", f"utn{index}"
        self.host_ip, self.ns_ip = f"10.200.{index}.1", f"10.200.{index}.2"
        self.cidr = f"10.200.{index}.0/30"
        self.tag = f"{TAG}:{self.ns}"
        self.link = {}
        self.captures = []

    def __enter__(self):
        try:
            self.create()
        except Exception:
            self.destroy()
            raise
        return self

    def __exit__(self, *exc):
        self.destroy()

    def cmd(self, args: list[str]) -> list[str]:
        return ["ip", "netns", "exec", self.ns] + list(args)

    def rule(self, table, chain, spec, position=None):
        where = ["-I", chain, str(position)] if position else ["-A", chain]
        sh(["iptables", "-w", "-t", table] + where + spec + ["-m", "comment", "--comment", self.tag])

    def create(self) -> None:
        self.destroy()
        up = uplink_mtu()
        out, mtu = up["uplink"], up["mtu"]
        sh(["sysctl", "-qw", "net.ipv4.ip_forward=1"])
        sh(["ip", "netns", "add", self.ns])
        sh(["ip", "link", "add", self.host_if, "type", "veth", "peer", "name", self.ns_if])
        sh(["ip", "link", "set", "dev", self.host_if, "mtu", str(mtu)])
        sh(["ip", "link", "set", "dev", self.ns_if, "mtu", str(mtu)])
        sh(["ip", "link", "set", self.ns_if, "netns", self.ns])
        sh(["ip", "addr", "add", f"{self.host_ip}/30", "dev", self.host_if])
        sh(["ip", "link", "set", self.host_if, "up"])
        sh(self.cmd(["ip", "addr", "add", f"{self.ns_ip}/30", "dev", self.ns_if]))
        sh(self.cmd(["ip", "link", "set", self.ns_if, "up"]))
        off = [w for k in OFFLOADS for w in (k, "off")]
        sh(["ethtool", "-K", self.host_if] + off)
        sh(self.cmd(["ethtool", "-K", self.ns_if] + off))
        self.link = dict(up, host_if=self.host_if, host_if_mtu=link_mtu(self.host_if), ns_if=self.ns_if,
                         ns_if_mtu=link_mtu(self.ns_if, self.ns),
                         offloads={self.host_if: link_offloads(self.host_if),
                                   self.ns_if: link_offloads(self.ns_if, self.ns)})
        if not self.link["host_if_mtu"] == self.link["ns_if_mtu"] == mtu:
            raise RuntimeError(f"veth MTU differs from the uplink MTU: {self.link}")
        if not all(offloads_off(v) for v in self.link["offloads"].values()):
            raise RuntimeError(f"offloads still on: {self.link['offloads']}")
        sh(self.cmd(["ip", "link", "set", "lo", "up"]))
        sh(self.cmd(["ip", "route", "add", "default", "via", self.host_ip]))
        sh(self.cmd(["sysctl", "-qw", "net.ipv6.conf.all.disable_ipv6=1"]), check=False)
        sh(self.cmd(["sysctl", "-qw", "net.ipv6.conf.default.disable_ipv6=1"]), check=False)
        os.makedirs(f"/etc/netns/{self.ns}", exist_ok=True)
        with open(f"/etc/netns/{self.ns}/resolv.conf", "w") as f:
            f.write("".join(f"nameserver {r}\n" for r in RESOLVERS))
        # rules are inserted at the top of their chains in this order, so they are evaluated before any VM rule
        position = 1
        for net in BLOCKED:
            self.rule("filter", "FORWARD", ["-i", self.host_if, "-d", net, "-j", "DROP"], position)
            position += 1
        self.rule("filter", "FORWARD", ["-i", self.host_if, "-o", out, "-j", "ACCEPT"], position)
        self.rule("filter", "FORWARD", ["-i", out, "-o", self.host_if, "-m", "conntrack", "--ctstate",
                                        "ESTABLISHED,RELATED", "-j", "ACCEPT"], position + 1)
        self.rule("filter", "FORWARD", ["-i", self.host_if, "-j", "DROP"], position + 2)
        self.rule("filter", "FORWARD", ["-o", self.host_if, "-j", "DROP"], position + 3)
        self.rule("filter", "INPUT", ["-i", self.host_if, "-j", "DROP"], 1)
        self.rule("nat", "POSTROUTING", ["-s", self.cidr, "-o", out, "-j", "MASQUERADE"], 1)

    def metadata(self) -> dict:
        """Run network settings and the counts of the run's stopped captures, recorded with every run."""
        return dict(ns=self.ns, subnet=self.cidr, ns_ip=self.ns_ip, host_ip=self.host_ip, **self.link,
                    captures=[c.stats() for c in self.captures if c.proc.poll() is not None])

    def write_metadata(self, path: str) -> dict:
        meta = self.metadata()
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        with open(path, "w") as f:
            json.dump(meta, f, indent=1)
        return meta

    def destroy(self) -> None:
        remove_tagged_rules(self.tag)
        sh(["ip", "netns", "del", self.ns], check=False)
        sh(["ip", "link", "del", self.host_if], check=False)
        shutil.rmtree(f"/etc/netns/{self.ns}", ignore_errors=True)

    def capture(self, path: str, timeout: float = 10.0) -> Capture:
        """Start tcpdump on the run's own interface; returns once tcpdump reports that it is listening."""
        if not path.endswith(".pcap"):
            raise ValueError("capture files end in .pcap (the tcpdump AppArmor profile allows that name)")
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        proc = subprocess.Popen(self.cmd(["tcpdump", "-i", self.ns_if, "-s", "0", "-U", "--immediate-mode", "-n",
                                          "-Z", "root", "-w", path]), stderr=subprocess.PIPE, text=True)
        cap = Capture(proc, path)
        self.captures.append(cap)
        if not cap.ready.wait(timeout) or proc.poll() is not None:
            cap.stop()
            raise RuntimeError(f"tcpdump did not start: {' | '.join(cap.lines)}")
        return cap

In [ ]:
%%writefile harness/vm_gate.py
"""VM gate for the benign collection testbed. Run as root on the VM, from the repository root, with the DeepSeek key
in DEEPSEEK_API_KEY (it is never written to disk):

    sudo -E python3 harness/vm_gate.py

Every check below runs on the VM:
  V0 environment     root, Python, the commands the testbed needs, agent versions, the key present
  V1 offline checks  harness/offline_check.py --agents (gateway, MCP tool log, both agents against a local fake
                     endpoint)
  V2 isolation       two namespaces at once: each capture holds only its own run's traffic; a run cannot reach another
                     run, the VM itself or the cloud metadata address
  V3 Anthropic API   inside a namespace, through the gateway: a streamed request with a tool definition is answered
                     (whether the model called the tool is reported); a long streamed answer reaches the client piece
                     by piece; three calls share one TLS connection on the wire
  V4 key log         the run's TLS key log decrypts its capture with tshark; decrypted request lengths equal the
                     gateway's logged lengths; without the key log nothing decrypts
  V5 Responses API   a streamed request with a function tool on DeepSeek's Responses API, the wire format Codex uses,
                     is answered as an event stream (whether the model called the tool is reported)
  V6 Claude Code     a one-tool task through the gateway, with the MCP tool log and a capture
  V7 Codex           the same task with Codex
  V8 fidelity        each agent runs the same task straight to DeepSeek; TLS connections to the model host, offered
                     ALPN protocols and upload bytes are compared with the gateway run
Gates: gate 1 (Codex on DeepSeek) = V5 and V7; gate 2 (gateway keeps the traffic shape) = V1, V3 and V8;
testbed = V0, V2, V4 and V6.

Upload bytes are measured three ways per TCP connection: raw TCP payload of every captured segment (resent segments
included), TCP payload counted once from the sequence numbers (resent bytes removed), and TLS record bytes (record
header plus record length, each record once). V4 and V8 report the last two and the bytes resent.

Wire checks on a run capture (V2 for both runs, V8 for all four agent runs); a run fails when any of them fails:
  - both veth ends have the uplink's MTU and have segmentation and receive offloads off (checked when the run's
    namespace is created, reported here);
  - no ICMP "fragmentation needed" message in the capture;
  - no captured IP packet larger than the MTU (a larger one is an offload block, not a wire packet);
  - upload bytes resent at most RESENT_TOLERANCE_SEGMENTS full-size segments (MTU minus 40 bytes each) over the whole
    capture; expected 0 on a clean link; download bytes resent by the endpoint are reported, not gated;
  - tcpdump wrote every packet its filter received and dropped none.
Every run directory gets net.json: uplink, MTU, offload states, subnet and capture counts.

Upstream drops: when a call through the gateway ends in an error on the endpoint's side (error_side "upstream" in the
gateway log) and the first FIN or RST on its connection in the capture comes from the endpoint, the endpoint dropped
the connection; this is recorded as an upstream drop, not a gateway fault. A check through the gateway (V3, V5, V6,
V7) that meets an upstream drop is run once more and fails only if the rerun meets one too. A call the agent itself
closed (error_side "agent") is reported but is not a gateway fault either.

Key logs are deleted when the gate ends. Captures and logs stay under data/harness/vm_gate/<time>/ (git-ignored).
"""
from __future__ import annotations

import argparse
import http.client
import json
import os
import platform
import re
import shutil
import socket
import ssl
import subprocess
import sys
import time
import urllib.error
import urllib.request

HERE = os.path.dirname(os.path.abspath(__file__))
REPO = os.path.dirname(HERE)
sys.path.insert(0, HERE)

MODEL_HOST = "api.deepseek.com"
ANTHROPIC_BASE = "https://api.deepseek.com/anthropic"
RESPONSES_BASE = "https://api.deepseek.com"
MODEL = "deepseek-flash"
WORD_COUNT_SCHEMA = {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}
AGENT_TIMEOUT = 300
NEEDED = ["ip", "iptables", "tcpdump", "tshark", "ethtool", "node", "npm", "claude", "codex", "openssl"]
RESENT_TOLERANCE_SEGMENTS = 1


# ---------------------------------------------------------------- clients run inside a namespace (internal mode)
def client_anthropic(port: int) -> dict:
    """Three calls on one kept-alive connection to the gateway: tool request, long streamed answer, short call."""
    key = os.environ["DEEPSEEK_API_KEY"]
    conn = http.client.HTTPConnection("127.0.0.1", port, timeout=300)
    headers = {"content-type": "application/json", "x-api-key": key, "anthropic-version": "2023-06-01"}
    out = {}

    def call(name, body, stream):
        t0 = time.time()
        conn.request("POST", "/v1/messages", json.dumps(body), headers)
        resp = conn.getresponse()
        reads, text = [], b""
        while True:
            piece = resp.read1(65536) if stream else resp.read()
            if not piece:
                break
            reads.append((time.time(), len(piece)))
            text += piece
            if not stream:
                break
        out[name] = dict(status=resp.status, t0=t0, reads=reads, bytes=len(text),
                         content_type=resp.getheader("content-type"),
                         head=text[:300].decode("utf-8", "replace") if resp.status != 200 else None)
        return text.decode("utf-8", "replace")

    text = call("tool", dict(model=MODEL, max_tokens=256, stream=True,
                             tools=[dict(name="word_count", description="Count the words in a text.",
                                         input_schema=WORD_COUNT_SCHEMA)],
                             messages=[dict(role="user", content="Use the word_count tool to count the words in: "
                                                                 "alpha beta gamma delta")]), True)
    out["tool"]["tool_use"] = bool(re.search(r'"type"\s*:\s*"tool_use"', text)
                                   and re.search(r'"name"\s*:\s*"word_count"', text))
    call("stream", dict(model=MODEL, max_tokens=3000, stream=True, messages=[dict(
        role="user", content="Write the whole numbers from 1 to 400 in order, separated by single spaces, "
                             "and nothing else.")]), True)
    call("short", dict(model=MODEL, max_tokens=16, messages=[dict(role="user", content="Reply with: ok")]), False)
    conn.close()
    return out


def client_responses(port: int) -> dict:
    key = os.environ["DEEPSEEK_API_KEY"]
    conn = http.client.HTTPConnection("127.0.0.1", port, timeout=300)
    body = dict(model=MODEL, stream=True, input="Use the word_count tool to count the words in: alpha beta gamma delta",
                tools=[dict(type="function", name="word_count", description="Count the words in a text.",
                            parameters=WORD_COUNT_SCHEMA)])
    conn.request("POST", "/responses", json.dumps(body),
                 {"content-type": "application/json", "authorization": f"Bearer {key}"})
    resp = conn.getresponse()
    text = b""
    while True:
        piece = resp.read1(65536)
        if not piece:
            break
        text += piece
    conn.close()
    t = text.decode("utf-8", "replace")
    return dict(status=resp.status, content_type=resp.getheader("content-type"), bytes=len(text),
                function_call=bool(re.search(r'"type"\s*:\s*"function_call"', t)
                                   and re.search(r'"name"\s*:\s*"word_count"', t)),
                events=sorted(set(re.findall(r"event:\s*(\S+)", t))),
                head=t[:300] if resp.status != 200 else None)


def client_fetch(url: str) -> dict:
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    try:
        with opener.open(url, timeout=30) as r:
            return dict(ok=True, status=r.status)
    except urllib.error.HTTPError as e:
        return dict(ok=True, status=e.code)
    except Exception as e:
        return dict(ok=False, error=f"{type(e).__name__}: {e}")


def client_reach(ip: str, port: int) -> dict:
    s = socket.socket()
    s.settimeout(4)
    try:
        s.connect((ip, port))
        return dict(result="connected")
    except ConnectionRefusedError:
        return dict(result="refused")
    except (socket.timeout, TimeoutError):
        return dict(result="timeout")
    except OSError as e:
        return dict(result=f"error {e.errno}")
    finally:
        s.close()


def client_alpn(host: str) -> dict:
    ctx = ssl.create_default_context()
    ctx.set_alpn_protocols(["h2", "http/1.1"])
    with socket.create_connection((host, 443), timeout=15) as raw, ctx.wrap_socket(raw, server_hostname=host) as s:
        return dict(selected=s.selected_alpn_protocol(), tls=s.version())


# ---------------------------------------------------------------- helpers on the VM side
def sh(args, timeout=60, **kw):
    return subprocess.run(args, capture_output=True, text=True, timeout=timeout, **kw)


def in_ns(net, client: str, *args, timeout=400) -> dict:
    r = sh(net.cmd([sys.executable, os.path.abspath(__file__), "--client", client, *map(str, args)]), timeout=timeout)
    try:
        return json.loads(r.stdout.strip().splitlines()[-1])
    except (ValueError, IndexError):
        return dict(error=f"client {client} failed: {r.stderr.strip()[-400:]}")


def tshark_rows(pcap, display_filter, fields, keylog=None):
    args = ["tshark", "-r", pcap, "-n"]
    if keylog:
        args += ["-o", f"tls.keylog_file:{keylog}"]
    args += ["-Y", display_filter, "-T", "fields", "-E", "separator=\t"]
    for f in fields:
        args += ["-e", f]
    r = sh(args, timeout=300)
    return [ln.split("\t") for ln in r.stdout.splitlines() if ln.strip()]


NOT_RESENT = ("!tcp.analysis.retransmission && !tcp.analysis.fast_retransmission "
              "&& !tcp.analysis.spurious_retransmission")


def cols(row, n):
    return (list(row) + [""] * n)[:n]


def truthy(value: str) -> bool:
    return value.strip().lower() in ("1", "true")


def epoch(value: str) -> float:
    """frame.time_epoch as seconds; tshark prints a number, some versions an ISO 8601 time."""
    try:
        return float(value)
    except ValueError:
        import datetime
        v = value.strip().replace("Z", "+00:00")
        if "." in v:
            head, frac = v.split(".", 1)
            n = len(frac) - len(frac.lstrip("0123456789"))
            v = f"{head}.{frac[:min(n, 6)]}{frac[n:]}"
        t = datetime.datetime.fromisoformat(v)
        if t.tzinfo is None:
            t = t.replace(tzinfo=datetime.timezone.utc)
        return t.timestamp()


def upload_measures(segments, records, client_ip, flagged=()) -> dict:
    """Bytes per TCP stream, from tshark rows.

    segments  rows (tcp.stream, ip.src, tcp.seq, tcp.len), one per captured segment with payload
    records   rows (tcp.stream, ip.src, tls.record.length values joined by commas), frames that are not resends
    flagged   rows (tcp.stream, ip.src, tcp.len), segments tshark marks as resent
    Per stream:
      up_raw    client payload of every captured segment, resent segments included
      up_tcp    client payload counted once: highest sequence end minus lowest sequence start
      resent    up_raw minus up_tcp
      up_tls    client TLS record bytes: 5-byte record header plus record length, each record once
      down_raw  endpoint payload of every captured segment
      down_resent  endpoint payload of segments tshark marks as resent (a segment the endpoint lost on the way and
                   sent again is seen once in a client-side capture, so the sequence span cannot count it)
    The capture is taken on the client's side of each connection, so every byte the client sent appears there before
    any resend of it; up_tls and up_tcp therefore count the same bytes and should be equal.
    """
    empty = dict(up_raw=0, up_tcp=0, resent=0, up_tls=0, down_raw=0, down_resent=0)
    out, span = {}, {}
    for row in segments:
        stream, src, seq, length = cols(row, 4)
        if not seq.isdigit() or not length.isdigit():
            continue
        f = out.setdefault(stream, dict(empty))
        n, start = int(length), int(seq)
        if src == client_ip:
            f["up_raw"] += n
            lo, hi = span.get(stream, (start, start + n))
            span[stream] = (min(lo, start), max(hi, start + n))
        else:
            f["down_raw"] += n
    for stream, (lo, hi) in span.items():
        out[stream]["up_tcp"] = hi - lo
        out[stream]["resent"] = out[stream]["up_raw"] - (hi - lo)
    for row in records:
        stream, src, lengths = cols(row, 3)
        if src == client_ip and lengths:
            f = out.setdefault(stream, dict(empty))
            f["up_tls"] += sum(5 + int(x) for x in lengths.split(",") if x.strip().isdigit())
    for row in flagged:
        stream, src, length = cols(row, 3)
        if src != client_ip and length.isdigit():
            out.setdefault(stream, dict(empty))["down_resent"] += int(length)
    return out


def flows(pcap: str, client_ip: str) -> dict:
    """Per TCP stream: server name, offered ALPN list, selected ALPN when visible, and the byte counts of
    upload_measures."""
    out = {}
    for row in tshark_rows(pcap, "tls.handshake.type == 1", ["tcp.stream", "tls.handshake.extensions_server_name",
                                                              "tls.handshake.extensions_alpn_str"]):
        stream, sni, alpn = cols(row, 3)
        out.setdefault(stream, {}).update(sni=sni, offered=alpn)
    for row in tshark_rows(pcap, "tls.handshake.type == 2", ["tcp.stream", "tls.handshake.extensions_alpn_str"]):
        stream, alpn = cols(row, 2)
        if alpn:
            out.setdefault(stream, {})["selected"] = alpn
    segments = tshark_rows(pcap, "tcp.len > 0", ["tcp.stream", "ip.src", "tcp.seq", "tcp.len"])
    records = tshark_rows(pcap, f"tls.record && {NOT_RESENT}", ["tcp.stream", "ip.src", "tls.record.length"])
    flagged = tshark_rows(pcap, f"tcp.len > 0 && !({NOT_RESENT})", ["tcp.stream", "ip.src", "tcp.len"])
    for stream, m in upload_measures(segments, records, client_ip, flagged).items():
        out.setdefault(stream, {}).update(m)
    return out


def first(value: str) -> str:
    """Outer header value of a field that ICMP errors repeat for the quoted inner header."""
    return value.split(",")[0]


def wire_facts(pcap: str, fl: dict, meta: dict) -> dict:
    """Capture-level facts for wire_verdict: ICMP fragmentation-needed messages, largest packet, resent bytes."""
    icmp = tshark_rows(pcap, "icmp.type == 3 && icmp.code == 4", ["ip.src", "icmp.mtu"])
    sizes = [int(first(r[0])) for r in tshark_rows(pcap, "ip", ["ip.len"]) if r and first(r[0]).isdigit()]
    cap = next((c for c in meta.get("captures", []) if c["path"] == pcap), {})
    return dict(mtu=meta.get("mtu"), uplink=meta.get("uplink"), uplink_mtu=meta.get("uplink_mtu"),
                host_if_mtu=meta.get("host_if_mtu"), ns_if_mtu=meta.get("ns_if_mtu"),
                offloads=meta.get("offloads", {}), icmp_frag_needed=len(icmp),
                icmp_next_hop_mtu=sorted({first(cols(r, 2)[1]) for r in icmp}), packets=len(sizes),
                largest_packet=max(sizes, default=0), up_resent=sum(f.get("resent", 0) for f in fl.values()),
                down_resent=sum(f.get("down_resent", 0) for f in fl.values()),
                capture={k: cap.get(k) for k in ("captured", "received", "dropped_kernel", "dropped_interface",
                                                 "complete")})


def wire_verdict(w: dict) -> tuple[bool, list[str]]:
    """Pass or fail of a run capture, with the reasons for a failure."""
    import netns
    mtu = w.get("mtu") or 0
    tolerance = RESENT_TOLERANCE_SEGMENTS * max(mtu - 40, 0)
    fails = []
    if not mtu or not w.get("host_if_mtu") == w.get("ns_if_mtu") == mtu:
        fails.append(f"veth MTU {w.get('host_if_mtu')}/{w.get('ns_if_mtu')} is not the uplink MTU {mtu}")
    if not w.get("offloads") or not all(netns.offloads_off(v) for v in w["offloads"].values()):
        fails.append(f"offloads not all off: {w.get('offloads')}")
    if w["icmp_frag_needed"]:
        fails.append(f"{w['icmp_frag_needed']} ICMP fragmentation-needed messages (next-hop MTU "
                     f"{', '.join(w['icmp_next_hop_mtu'])})")
    if w["largest_packet"] > mtu:
        fails.append(f"captured packet of {w['largest_packet']} B is larger than the MTU")
    if w["up_resent"] > tolerance:
        fails.append(f"{w['up_resent']} upload bytes resent, above the tolerance of {tolerance} B")
    if not w["capture"].get("complete"):
        fails.append(f"capture incomplete {w['capture']}")
    return not fails, fails


def describe_wire(w: dict) -> str:
    import netns
    ok, fails = wire_verdict(w)
    c = w["capture"]
    off = bool(w["offloads"]) and all(netns.offloads_off(v) for v in w["offloads"].values())
    return (f"MTU {w['mtu']} (uplink {w['uplink']} {w['uplink_mtu']}, veth {w['host_if_mtu']}/{w['ns_if_mtu']}), "
            f"offloads off {off}, "
            f"ICMP fragmentation-needed {w['icmp_frag_needed']}, largest packet {w['largest_packet']} B, upload "
            f"resent {w['up_resent']} B, download resent {w['down_resent']} B, tcpdump captured {c.get('captured')} "
            f"of {c.get('received')} (dropped {c.get('dropped_kernel')})"
            + ("" if ok else f" [FAIL: {'; '.join(fails)}]"))


def close_rows_to_events(rows, client_ip) -> dict:
    """FIN and RST packets per client port, in capture order: {port: [(time, sent by the endpoint, 'FIN'/'RST')]}.
    rows: (frame.time_epoch, ip.src, tcp.srcport, tcp.dstport, tcp.flags.fin, tcp.flags.reset)."""
    events = {}
    for row in rows:
        t, src, sport, dport, fin, rst = cols(row, 6)
        from_endpoint = src != client_ip
        port = dport if from_endpoint else sport
        events.setdefault(port, []).append((epoch(t), from_endpoint, "RST" if truthy(rst) else "FIN"))
    return events


def close_events(pcap: str, client_ip: str) -> dict:
    rows = tshark_rows(pcap, "tcp.flags.fin == 1 || tcp.flags.reset == 1",
                       ["frame.time_epoch", "ip.src", "tcp.srcport", "tcp.dstport", "tcp.flags.fin",
                        "tcp.flags.reset"])
    return close_rows_to_events(rows, client_ip)


def upstream_drops(calls: list[dict], events: dict) -> list[dict]:
    """Calls through the gateway that ended in an error on the endpoint's side, on a connection the endpoint closed
    first."""
    drops = []
    for c in calls:
        if not c.get("error") or not c.get("up_local") or c.get("error_side", "upstream") != "upstream":
            continue
        port = c["up_local"].rsplit(":", 1)[1]
        after = [e for e in events.get(port, []) if e[0] >= c["t"] - 0.001]
        if not after or not after[0][1]:
            continue
        t, _, kind = after[0]
        if c.get("status") is None:
            phase = "before the response head"
        elif not c.get("resp_body_bytes"):
            phase = f"after the {c['status']} response head, before any body"
        else:
            phase = f"inside the body, after {c['resp_body_bytes']} bytes"
        drops.append(dict(conn=c["conn"], seq=c["seq"], path=c["path"].split("?")[0], status=c.get("status"),
                          phase=phase, closed_by=f"endpoint {kind}",
                          seconds_after_head=round(t - c["t_resp_head"], 3) if c.get("t_resp_head") else None,
                          gateway_error=c["error"]))
    return drops


def describe_drops(drops: list[dict]) -> str:
    return "; ".join(f"upstream drop on {d['path']} {d['phase']} ({d['closed_by']}"
                     + (f", {d['seconds_after_head']} s after the head" if d["seconds_after_head"] is not None
                        else "") + ")" for d in drops)


def model_flows(fl: dict) -> list[dict]:
    return [f for f in fl.values() if f.get("sni") == MODEL_HOST]


def answered(text: str) -> bool:
    """The one-tool task asks for the word count only; the answer must contain the number 4 and no other number."""
    return re.findall(r"\d+", text or "") == ["4"]


def read_jsonl(path, event=None):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        rows = [json.loads(x) for x in f if x.strip()]
    return [r for r in rows if event is None or r.get("event") == event]


class Gate:
    def __init__(self, out_dir, key):
        self.out = out_dir
        self.key = key
        self.results = []
        self.keylogs = []

    def record(self, check, passed, detail, data=None):
        self.results.append(dict(check=check, where="VM", passed=passed, detail=detail, data=data))
        mark = "PASS" if passed else ("SKIP" if passed is None else "FAIL")
        print(f"{mark}  {check}: {detail}", flush=True)

    def start_gateway(self, net, run_dir, name, upstream, port):
        keylog = os.path.join(run_dir, f"{name}.keys")
        log = os.path.join(run_dir, f"{name}_calls.jsonl")
        self.keylogs.append(keylog)
        proc = subprocess.Popen(net.cmd([sys.executable, os.path.join(HERE, "call_gateway.py"), "--listen",
                                         f"127.0.0.1:{port}", "--upstream", upstream, "--log", log,
                                         "--keylog", keylog]), stderr=subprocess.PIPE, text=True)
        line = proc.stderr.readline()
        if "listening" not in line:
            raise RuntimeError(f"gateway did not start: {line}")
        return proc, log, keylog

    @staticmethod
    def stop(proc):
        proc.terminate()
        try:
            proc.wait(10)
        except subprocess.TimeoutExpired:
            proc.kill()

    # V0
    def environment(self):
        versions, missing = {}, []
        for cmd in NEEDED:
            if not shutil.which(cmd):
                missing.append(cmd)
                continue
            flag = {"tshark": ["--version"], "tcpdump": ["--version"], "ip": ["-V"],
                    "openssl": ["version"]}.get(cmd, ["--version"])
            r = sh([cmd] + flag)
            versions[cmd] = (r.stdout or r.stderr).strip().splitlines()[0] if (r.stdout or r.stderr).strip() else "?"
        try:
            import netns
            up = netns.uplink_mtu()
        except Exception as e:
            up = dict(error=f"{type(e).__name__}: {e}")
        ok = os.geteuid() == 0 and sys.version_info >= (3, 9) and not missing and bool(self.key) and "mtu" in up
        self.record("V0 environment", ok,
                    f"root {os.geteuid() == 0}; Python {platform.python_version()}; {platform.platform()}; "
                    f"missing commands {missing or 'none'}; DeepSeek key present {bool(self.key)}; uplink {up}",
                    dict(versions=versions, uplink=up))
        return ok

    # V1
    def offline(self):
        d = os.path.join(self.out, "offline")
        r = sh([sys.executable, os.path.join(HERE, "offline_check.py"), "--agents", "--out", d], timeout=900)
        rep = json.load(open(os.path.join(d, "offline_check.json"))) if os.path.exists(
            os.path.join(d, "offline_check.json")) else []
        failed = [x["check"] for x in rep if x["passed"] is False]
        skipped = [x["check"] for x in rep if x["passed"] is None]
        ok = r.returncode == 0 and rep and not failed and not skipped
        self.record("V1 offline checks", ok, f"{len(rep)} checks; failed {failed or 'none'}; skipped "
                                             f"{skipped or 'none'}", dict(results=rep))

    # V2
    def isolation(self):
        import netns
        d = os.path.join(self.out, "isolation")
        with netns.RunNet(11) as a, netns.RunNet(12) as b:
            ca, cb = a.capture(os.path.join(d, "a.pcap")), b.capture(os.path.join(d, "b.pcap"))
            try:
                me = os.path.abspath(__file__)
                pa = subprocess.Popen(a.cmd([sys.executable, me, "--client", "fetch", f"https://{MODEL_HOST}/"]),
                                      stdout=subprocess.PIPE, text=True)
                pb = subprocess.Popen(b.cmd([sys.executable, me, "--client", "fetch", "https://www.example.com/"]),
                                      stdout=subprocess.PIPE, text=True)
                fa = json.loads((pa.communicate(timeout=60)[0].strip().splitlines() or ["{}"])[-1])
                fb = json.loads((pb.communicate(timeout=60)[0].strip().splitlines() or ["{}"])[-1])
                time.sleep(1)
            finally:
                ca.stop()
                cb.stop()
            meta_a = a.write_metadata(os.path.join(d, "a_net.json"))
            meta_b = b.write_metadata(os.path.join(d, "b_net.json"))
            # reachability is tested after the captures stop, so its probe packets are not part of the capture check
            reach_run = in_ns(a, "reach", b.ns_ip, 9)
            reach_vm = in_ns(a, "reach", a.host_ip, 22)
            reach_meta = in_ns(a, "reach", "169.254.169.254", 80)
            ra = tshark_rows(ca.path, "ip", ["ip.src", "ip.dst"])
            rb = tshark_rows(cb.path, "ip", ["ip.src", "ip.dst"])
            sni_a = {r[0] for r in tshark_rows(ca.path, "tls.handshake.type == 1",
                                               ["tls.handshake.extensions_server_name"])}
            sni_b = {r[0] for r in tshark_rows(cb.path, "tls.handshake.type == 1",
                                               ["tls.handshake.extensions_server_name"])}
            ips_a = {ip for row in ra for cell in row for ip in cell.split(",")}
            ips_b = {ip for row in rb for cell in row for ip in cell.split(",")}
            clean = (b.ns_ip not in ips_a and a.ns_ip not in ips_b and a.ns_ip in ips_a and b.ns_ip in ips_b)
            wa = wire_facts(ca.path, flows(ca.path, a.ns_ip), meta_a)
            wb = wire_facts(cb.path, flows(cb.path, b.ns_ip), meta_b)
            ok = (fa.get("ok") and fb.get("ok") and sni_a == {MODEL_HOST} and sni_b == {"www.example.com"} and clean
                  and reach_run.get("result") == "timeout" and reach_vm.get("result") == "timeout"
                  and reach_meta.get("result") == "timeout" and wire_verdict(wa)[0] and wire_verdict(wb)[0])
            self.record("V2 isolation", ok,
                        f"run A fetched {MODEL_HOST} ({fa}), run B fetched www.example.com ({fb}) at the same time; "
                        f"capture A: {len(ra)} packets, server names {sorted(sni_a)}; capture B: {len(rb)} packets, "
                        f"server names {sorted(sni_b)}; no packet of one run in the other's capture {clean}; "
                        f"A to B {reach_run}, A to the VM {reach_vm}, A to the metadata address {reach_meta}; "
                        f"wire A: {describe_wire(wa)}; wire B: {describe_wire(wb)}", dict(wire_a=wa, wire_b=wb))

    # V3 and V4
    def with_rerun(self, attempt, index, rerun_index, name):
        """Runs one attempt; when the endpoint dropped a call, records the drop and runs the attempt once more."""
        first = attempt(index, name)
        if not first["drops"]:
            return first, ""
        note = f"first attempt: {describe_drops(first['drops'])}; rerun once"
        second = attempt(rerun_index, name + "_rerun")
        second["first_drops"] = first["drops"]
        if second["drops"]:
            note += f"; rerun: {describe_drops(second['drops'])}, so the endpoint dropped a call again"
        else:
            note += "; the rerun had no upstream drop"
        return second, note

    def anthropic_attempt(self, index, name):
        import netns
        d = os.path.join(self.out, name)
        with netns.RunNet(index) as net:
            cap = net.capture(os.path.join(d, "run.pcap"))
            gw = None
            try:
                gw, log, keylog = self.start_gateway(net, d, "gw", ANTHROPIC_BASE, 8081)
                res = in_ns(net, "anthropic", 8081)
                time.sleep(0.5)
            finally:
                if gw:
                    self.stop(gw)
                time.sleep(1)
                cap.stop()
            meta = net.write_metadata(os.path.join(d, "net.json"))
            calls = read_jsonl(log, "call")
            return dict(res=res, calls=calls, flows=flows(cap.path, net.ns_ip), pcap=cap.path, keylog=keylog, net=meta,
                        drops=upstream_drops(calls, close_events(cap.path, net.ns_ip)))

    def anthropic_api(self):
        a, note = self.with_rerun(self.anthropic_attempt, 13, 23, "anthropic_api")
        lead = note + "; " if note else ""
        res, calls, fl = a["res"], a["calls"], a["flows"]
        data = dict(client=res, calls=calls, flows=fl, first_attempt_drops=a.get("first_drops"),
                    drops=a["drops"])
        if a["drops"]:
            self.record("V3 Anthropic API", False, note, data)
            self.record("V4 key log", False, "not run: the endpoint dropped a call on both attempts")
            return
        mf = model_flows(fl)
        if "error" in res or len(calls) != 3:
            self.record("V3 Anthropic API", False, f"{lead}client result {str(res)[:400]}; {len(calls)} logged calls",
                        data)
            self.record("V4 key log", False, "not run: V3 did not complete")
            return
        tool_ok = res["tool"]["status"] == 200
        s = res["stream"]
        g = calls[1]
        first_arrival = s["reads"][0][0] if s["reads"] else None
        span = (g["t_end"] - g["t_first_byte"]) if g["t_first_byte"] else 0
        lag = (first_arrival - g["t_first_byte"]) if first_arrival and g["t_first_byte"] else None
        streamed = (s["status"] == 200 and len(s["reads"]) >= 5 and lag is not None and lag < 0.25
                    and span >= 1.0 and first_arrival < g["t_end"] - 0.5)
        one_conn = len({c["up_conn"] for c in calls}) == 1 and [c["up_new"] for c in calls] == [True, False, False]
        ok = tool_ok and streamed and one_conn and len(mf) == 1 and all(c["status"] == 200 for c in calls)
        self.record("V3 Anthropic API", ok,
                    f"{lead}tool request {res['tool']['status']}, tool call returned {res['tool']['tool_use']}; long "
                    f"answer streamed over {span:.2f} s in {len(s['reads'])} reads, first piece reached the client "
                    f"{(lag or 0) * 1e3:.0f} ms after the gateway read it; statuses "
                    f"{[c['status'] for c in calls]}; one upstream connection for 3 calls {one_conn}; TLS "
                    f"connections to {MODEL_HOST} in the capture {len(mf)}"
                    + (f"; error body {res['tool']['head'] or s['head']}" if not tool_ok or s['status'] != 200
                       else ""), data)
        # V4: decrypt with the run's key log, then without it
        req = tshark_rows(a["pcap"], "http.request", ["http.request.method", "http.request.uri",
                                                      "http.content_length"], keylog=a["keylog"])
        rsp = tshark_rows(a["pcap"], "http.response", ["http.response.code"], keylog=a["keylog"])
        plain = tshark_rows(a["pcap"], "http", ["frame.number"])
        dec_lengths = [int(r[2]) for r in req if len(r) > 2 and r[2].isdigit()]
        logged = [c["req_body_bytes"] for c in calls]
        f = mf[0] if mf else {}
        framed = sum(c["req_head_bytes"] + c["req_wire_body_bytes"] for c in calls)
        ok4 = dec_lengths == logged and [r[0] for r in rsp] == ["200"] * 3 and not plain
        self.record("V4 key log", ok4,
                    f"with the key log tshark decrypted {len(req)} requests with lengths {dec_lengths} (gateway "
                    f"logged {logged}) and {len(rsp)} responses {[r[0] for r in rsp]}; without it "
                    f"{len(plain)} HTTP frames; upload on the wire {f.get('up_tls', 0)} bytes of TLS records and "
                    f"{f.get('up_tcp', 0)} TCP bytes without resends ({f.get('resent', 0)} bytes resent) for "
                    f"{framed} bytes of HTTP, so TLS overhead {f.get('up_tls', 0) - framed} bytes over 3 calls")

    # V5
    def responses_attempt(self, index, name):
        import netns
        d = os.path.join(self.out, name)
        with netns.RunNet(index) as net:
            cap = net.capture(os.path.join(d, "run.pcap"))
            gw = None
            try:
                gw, log, _ = self.start_gateway(net, d, "gw", RESPONSES_BASE, 8082)
                res = in_ns(net, "responses", 8082)
                alpn = in_ns(net, "alpn", MODEL_HOST)
            finally:
                if gw:
                    self.stop(gw)
                time.sleep(1)
                cap.stop()
            meta = net.write_metadata(os.path.join(d, "net.json"))
            calls = read_jsonl(log, "call")
            return dict(res=res, alpn=alpn, calls=calls, pcap=cap.path, net=meta,
                        drops=upstream_drops(calls, close_events(cap.path, net.ns_ip)))

    def responses_api(self):
        a, note = self.with_rerun(self.responses_attempt, 14, 24, "responses_api")
        self.alpn = a["alpn"]
        res, calls = a["res"], a["calls"]
        data = dict(client=res, calls=calls, first_attempt_drops=a.get("first_drops"), drops=a["drops"])
        if a["drops"]:
            self.record("V5 Responses API", False, note, data)
            return
        ok = res.get("status") == 200 and len(calls) == 1 \
            and (calls[0]["resp_content_type"] or "").startswith("text/event-stream")
        self.record("V5 Responses API", ok,
                    (note + "; " if note else "")
                    + f"POST /responses {res.get('status')}, content type {res.get('content_type')}, function call "
                    f"to word_count {res.get('function_call')}, events {res.get('events')}"
                    + (f"; error body {res.get('head') or res.get('error')}" if not ok else ""), data)

    # V6 to V8
    def agent_run(self, which, mode, index):
        """Gateway runs get the upstream-drop rerun; direct runs have no gateway log to read drops from."""
        if mode == "gateway":
            return self.with_rerun(lambda i, name: self.agent(which, mode, i, name), index, index + 10,
                                   f"{which}_{mode}")
        return self.agent(which, mode, index, f"{which}_{mode}"), ""

    def agent(self, which, mode, index, name):
        import netns
        from offline_check import AGENT_PROMPT, MCP, codex_config
        d = os.path.join(self.out, name)
        home, ws = os.path.join(d, "home"), os.path.join(d, "ws")
        os.makedirs(home, exist_ok=True)
        os.makedirs(ws, exist_ok=True)
        tool_log = os.path.join(d, "tools.jsonl")
        mcp = {"command": sys.executable, "args": [MCP], "env": {"UT_TOOL_LOG": tool_log, "UT_WORKSPACE": ws}}
        env = {k: v for k, v in os.environ.items() if not k.startswith(("ANTHROPIC_", "OPENAI_", "CODEX_", "CLAUDE_"))
               and k.lower() not in ("http_proxy", "https_proxy", "all_proxy")}
        env.update(HOME=home, UT_TOOL_LOG=tool_log, UT_WORKSPACE=ws)
        with netns.RunNet(index) as net:
            cap = net.capture(os.path.join(d, "run.pcap"))
            gw = log = None
            if which == "claude":
                if mode == "gateway":
                    gw, log, _ = self.start_gateway(net, d, "gw", ANTHROPIC_BASE, 8081)
                base = "http://127.0.0.1:8081" if mode == "gateway" else ANTHROPIC_BASE
                cfg = os.path.join(d, "mcp.json")
                with open(cfg, "w") as f:
                    json.dump({"mcpServers": {"bench": dict(mcp, type="stdio")}}, f)
                env.update(ANTHROPIC_BASE_URL=base, ANTHROPIC_AUTH_TOKEN=self.key,
                           ANTHROPIC_MODEL="deepseek-flash[1m]", ANTHROPIC_DEFAULT_OPUS_MODEL="deepseek-flash[1m]",
                           ANTHROPIC_DEFAULT_SONNET_MODEL="deepseek-flash[1m]",
                           ANTHROPIC_DEFAULT_HAIKU_MODEL="deepseek-flash")
                cmd = ["claude", "-p", AGENT_PROMPT, "--mcp-config", cfg, "--strict-mcp-config", "--allowedTools",
                       "mcp__bench__word_count", "--output-format", "json", "--max-turns", "6"]
            else:
                if mode == "gateway":
                    gw, log, _ = self.start_gateway(net, d, "gw", RESPONSES_BASE, 8082)
                base = "http://127.0.0.1:8082" if mode == "gateway" else RESPONSES_BASE + "/"
                codex_home = os.path.join(home, ".codex")
                os.makedirs(codex_home, exist_ok=True)
                with open(os.path.join(codex_home, "config.toml"), "w") as f:
                    f.write(codex_config(base, mcp))
                env.update(CODEX_HOME=codex_home, DEEPSEEK_API_KEY=self.key)
                cmd = ["codex", "exec", "--skip-git-repo-check", "--ephemeral", "-C", ws, "-o",
                       os.path.join(d, "last_message.txt"), AGENT_PROMPT]
            t0 = time.time()
            try:
                p = subprocess.run(net.cmd(cmd), env=env, cwd=ws, stdin=subprocess.DEVNULL, capture_output=True,
                                   text=True, timeout=AGENT_TIMEOUT)
                code, out, err = p.returncode, p.stdout, p.stderr
            except subprocess.TimeoutExpired as e:
                code, out, err = "timeout", str(e.stdout or ""), str(e.stderr or "")
            finally:
                time.sleep(1)
                if gw:
                    self.stop(gw)
                cap.stop()
            took = time.time() - t0
            with open(os.path.join(d, "agent_stdout.txt"), "w") as f:
                f.write(out)
            with open(os.path.join(d, "agent_stderr.txt"), "w") as f:
                f.write(err)
            meta = net.write_metadata(os.path.join(d, "net.json"))
            fl = flows(cap.path, net.ns_ip)
            events = close_events(cap.path, net.ns_ip)
            wire = wire_facts(cap.path, fl, meta)
        answer = ""
        if which == "claude":
            try:
                answer = str(json.loads(out).get("result", ""))
            except ValueError:
                pass
        elif os.path.exists(os.path.join(d, "last_message.txt")):
            with open(os.path.join(d, "last_message.txt")) as f:
                answer = f.read()
        calls = read_jsonl(log, "call") if log else []
        tools = read_jsonl(tool_log, "tool_call")
        mf = model_flows(fl)
        return dict(which=which, mode=mode, exit=code, seconds=round(took, 1), answer=answer.strip()[-200:],
                    calls=calls, tools=tools, model_flows=mf, pcap=cap.path, drops=upstream_drops(calls, events),
                    wire=wire,
                    other_hosts=sorted({f.get("sni") for f in fl.values()
                                        if f.get("sni") and f.get("sni") != MODEL_HOST}),
                    stderr_tail=err.strip()[-400:])

    def agent_check(self, name, r, note=""):
        if r["drops"]:
            self.record(name, False, note, r)
            return
        model_calls = [c for c in r["calls"] if c["path"].split("?")[0].endswith(("/v1/messages", "/responses"))]
        streamed = [c for c in model_calls if (c["resp_content_type"] or "").startswith("text/event-stream")]
        agent_conns = {c["conn"] for c in r["calls"]}
        up_conns = {c["up_conn"] for c in r["calls"] if c["up_conn"]}
        agent_closes = [c for c in r["calls"] if c["error"] and c.get("error_side") == "agent"]
        errors = [c["error"] for c in r["calls"] if c["error"] and c.get("error_side") != "agent"]
        ok = (r["exit"] == 0 and answered(r["answer"]) and len(model_calls) >= 2 and streamed
              and all(c["status"] == 200 for c in model_calls) and r["tools"] and r["tools"][0]["tool"] == "word_count"
              and len(agent_conns) == len(up_conns) and not errors and r["model_flows"])
        self.record(name, ok,
                    (note + "; " if note else "")
                    + f"exit {r['exit']} in {r['seconds']} s, answer {r['answer'][-40:]!r}; {len(model_calls)} model "
                    f"calls (statuses {[c['status'] for c in model_calls]}, {len(streamed)} streamed, request bytes "
                    f"{[c['req_body_bytes'] for c in model_calls]}); tool calls "
                    f"{[(t['tool'], t['args_bytes']) for t in r['tools']]}; "
                    f"agent connections {len(agent_conns)}, upstream connections {len(up_conns)}; calls the agent "
                    f"closed itself {len(agent_closes)}; gateway or endpoint errors {errors}; TLS connections "
                    f"to {MODEL_HOST} {len(r['model_flows'])}; other hosts {r['other_hosts']}"
                    + (f"; stderr {r['stderr_tail'][-300:]}" if not ok else ""), r)

    def fidelity(self, direct, gateway, label):
        def summary(r):
            mf = r["model_flows"]
            total = lambda k: sum(f.get(k, 0) for f in mf)
            return dict(connections=len(mf), offered=sorted({f.get("offered", "") for f in mf}),
                        selected=sorted({f.get("selected", "") for f in mf if f.get("selected")}),
                        up_tls=total("up_tls"), up_tcp=total("up_tcp"), up_raw=total("up_raw"),
                        resent=total("resent"), tool_calls=len(r["tools"]), exit=r["exit"],
                        answer=r["answer"][-20:])

        def pct(g, d):
            return f"{(g - d) / d * 100:+.1f}%" if d else "n/a"
        sd, sg = summary(direct), summary(gateway)
        wire_ok = wire_verdict(direct["wire"])[0] and wire_verdict(gateway["wire"])[0]
        probe = getattr(self, "alpn", None) or {}
        server_h2 = probe.get("selected") == "h2"
        direct_h2 = any("h2" in o.split(",") for o in sd["offered"]) and server_h2
        same_protocol = not direct_h2   # the gateway speaks HTTP/1.1 upstream
        same_conns = sd["connections"] == sg["connections"]
        ok = (direct["exit"] == 0 and gateway["exit"] == 0 and "selected" in probe and same_protocol and same_conns
              and sd["connections"] > 0 and wire_ok)
        self.record(f"V8 fidelity {label}", ok,
                    (f"ALPN probe of {MODEL_HOST} failed {probe}; " if "selected" not in probe else "")
                    + ("the gateway run is the rerun after an upstream drop; " if gateway.get("first_drops") else "")
                    + f"TLS connections to {MODEL_HOST}: straight to DeepSeek {sd['connections']}, through the "
                    f"gateway {sg['connections']}; offered ALPN: straight {sd['offered']}, gateway {sg['offered']}; "
                    f"upload as TLS records: straight {sd['up_tls']} B, gateway {sg['up_tls']} B "
                    f"({pct(sg['up_tls'], sd['up_tls'])}); upload as TCP bytes without resends: straight "
                    f"{sd['up_tcp']} B, gateway {sg['up_tcp']} B ({pct(sg['up_tcp'], sd['up_tcp'])}); resent: "
                    f"straight {sd['resent']} B, gateway {sg['resent']} B; tool calls: straight {sd['tool_calls']}, "
                    f"gateway {sg['tool_calls']}; {MODEL_HOST} picks h2 when offered: {server_h2}; same HTTP "
                    f"version on the wire {same_protocol}; same number of connections {same_conns}; wire straight: "
                    f"{describe_wire(direct['wire'])}; wire through the gateway: {describe_wire(gateway['wire'])}",
                    dict(direct=sd, gateway=sg, wire_direct=direct["wire"], wire_gateway=gateway["wire"]))

    def guard(self, name, fn, *args):
        try:
            return fn(*args)
        except Exception as e:
            self.record(name, False, f"stopped with {type(e).__name__}: {e}")
            return None

    def delete_keylogs(self):
        removed = 0
        for root, _, files in os.walk(self.out):
            for f in files:
                if f.endswith(".keys"):
                    os.remove(os.path.join(root, f))
                    removed += 1
        return removed


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--client", default=None, help=argparse.SUPPRESS)
    ap.add_argument("args", nargs="*", help=argparse.SUPPRESS)
    a = ap.parse_args(argv)
    if a.client:
        fn = dict(anthropic=lambda: client_anthropic(int(a.args[0])),
                  responses=lambda: client_responses(int(a.args[0])),
                  fetch=lambda: client_fetch(a.args[0]), reach=lambda: client_reach(a.args[0], int(a.args[1])),
                  alpn=lambda: client_alpn(a.args[0]))[a.client]
        try:
            print(json.dumps(fn()))
        except Exception as e:
            print(json.dumps(dict(error=f"{type(e).__name__}: {e}")))
        return 0

    key = os.environ.get("DEEPSEEK_API_KEY", "")
    if not key and sys.stdin.isatty():
        import getpass
        key = getpass.getpass("DeepSeek API key: ")
        os.environ["DEEPSEEK_API_KEY"] = key
    stamp = time.strftime("%Y%m%d_%H%M%S")
    out = os.path.join(REPO, "data", "harness", "vm_gate", stamp)
    os.makedirs(out, exist_ok=True)
    g = Gate(out, key)
    print(f"VM gate {stamp}, output {out}\n", flush=True)
    try:
        if not g.environment():
            print("\nV0 failed; the other checks need it")
            return 1
        import netns
        netns.cleanup_all()
        g.guard("V1 offline checks", g.offline)
        g.guard("V2 isolation", g.isolation)
        g.guard("V3 Anthropic API", g.anthropic_api)
        g.guard("V5 Responses API", g.responses_api)
        runs = {}
        for i, (which, mode) in enumerate([("claude", "gateway"), ("codex", "gateway"), ("claude", "direct"),
                                           ("codex", "direct")]):
            runs[(which, mode)] = g.guard(f"{which} {mode} run", g.agent_run, which, mode, 15 + i) or (None, "")
        for name, run_key in (("V6 Claude Code", ("claude", "gateway")), ("V7 Codex", ("codex", "gateway"))):
            run, note = runs[run_key]
            if run:
                g.agent_check(name, run, note)
            else:
                g.record(name, False, "the run did not complete")
        for label, which in (("Claude Code", "claude"), ("Codex", "codex")):
            direct, gateway = runs[(which, "direct")][0], runs[(which, "gateway")][0]
            if direct and gateway and not gateway["drops"]:
                g.fidelity(direct, gateway, label)
            else:
                g.record(f"V8 fidelity {label}", False, "a run did not complete or the endpoint dropped a call on "
                                                         "both gateway attempts")
    finally:
        removed = g.delete_keylogs()
        try:
            import netns
            netns.cleanup_all()
        except Exception:
            pass
        print(f"\nkey logs deleted: {removed}")
    passed = {r["check"].split()[0]: r["passed"] for r in g.results}
    fid = [r["passed"] for r in g.results if r["check"].startswith("V8")]
    gates = {
        "GATE 1 Codex on DeepSeek (V5, V7)": all(passed.get(k) for k in ("V5", "V7")),
        "GATE 2 gateway keeps the traffic shape (V1, V3, V8)": all(passed.get(k) for k in ("V1", "V3")) and fid
        and all(fid),
        "TESTBED isolation, capture, key log, Claude Code (V0, V2, V4, V6)": all(
            passed.get(k) for k in ("V0", "V2", "V4", "V6")),
    }
    print()
    for name, ok in gates.items():
        print(f"{'PASS' if ok else 'FAIL'}  {name}")
    with open(os.path.join(out, "vm_gate.json"), "w") as f:
        json.dump(dict(stamp=stamp, gates=gates, results=g.results), f, indent=1, default=str)
    print(f"\nreport {os.path.join(out, 'vm_gate.json')}")
    return 0 if all(gates.values()) else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile harness/vm_setup.sh
#!/usr/bin/env bash
# Installs what the testbed needs on a fresh Ubuntu 24.04 VM (x86_64 or arm64). Run once as root from the repository
# root: sudo bash harness/vm_setup.sh
# The agent versions are pinned to the versions the offline checks were run with.
set -euo pipefail

NODE_VERSION=22.22.0
CLAUDE_CODE_VERSION=2.1.302
CODEX_VERSION=0.163.0

if [ "$(id -u)" -ne 0 ]; then echo "run as root"; exit 1; fi
export DEBIAN_FRONTEND=noninteractive
apt-get update -q
apt-get install -y -q python3 iproute2 iptables tcpdump tshark openssl curl ca-certificates xz-utils git bubblewrap ethtool

case "$(uname -m)" in
  x86_64) arch=x64 ;;
  aarch64) arch=arm64 ;;
  *) echo "unsupported machine $(uname -m)"; exit 1 ;;
esac

# Node.js from the official release, checked against the published SHA-256 list (the Ubuntu package is too old for
# the Claude Code npm package, which requires Node.js 22)
if [ "$(node --version 2>/dev/null || true)" != "v${NODE_VERSION}" ]; then
  tmp="$(mktemp -d)"
  file="node-v${NODE_VERSION}-linux-${arch}.tar.xz"
  curl -fsSL "https://nodejs.org/dist/v${NODE_VERSION}/${file}" -o "${tmp}/${file}"
  curl -fsSL "https://nodejs.org/dist/v${NODE_VERSION}/SHASUMS256.txt" -o "${tmp}/SHASUMS256.txt"
  (cd "${tmp}" && grep " ${file}\$" SHASUMS256.txt | sha256sum -c -)
  tar -xJf "${tmp}/${file}" -C /usr/local --strip-components=1
  rm -rf "${tmp}"
fi

npm install -g --no-audit --no-fund "@anthropic-ai/claude-code@${CLAUDE_CODE_VERSION}" "@openai/codex@${CODEX_VERSION}"

echo "python3  $(python3 --version)"
echo "node     $(node --version)"
echo "claude   $(claude --version)"
echo "codex    $(codex --version)"
echo "tcpdump  $(tcpdump --version 2>&1 | head -1)"
echo "tshark   $(tshark --version 2>/dev/null | head -1)"
echo "ethtool  $(ethtool --version)"

In [ ]:
%%writefile harness/README.md
# Collection harness (benign runs)

Testbed that runs benign agent tasks and records, for every clean run, the exact byte length of each model call and
each tool call, next to a packet capture of that run. It runs on a Linux VM with root, because packet capture and
per-run network isolation need control of the operating system. Everything after collection runs in Colab.

| File | Role |
|---|---|
| `call_gateway.py` | Gateway the agent calls instead of the model endpoint. Logs the exact request and response body length of every call (JSON lines), forwards the request unchanged apart from Host and the upstream path prefix, streams the response back as it arrives, keeps one upstream connection per agent connection. Upstream TLS keys go to a key log for ground-truth labelling only. |
| `mcp_tools.py` | Local MCP tool server (stdio) with benign workspace tools. Logs each tool call's argument size at the tool boundary. |
| `netns.py` | One network namespace per run, with NAT, public DNS, containment rules and a capture on the run's own interface. The veth pair takes the uplink's MTU and has segmentation and receive offloads off, and tcpdump writes every packet as it arrives, so the capture holds wire-size packets and no packet is lost when it stops. |
| `offline_check.py` | Checks that need no network or root: exact sizes from 0 B to 1 MB, unbuffered streaming, connection reuse, reconnects, TLS key log, MCP tool log, and (with `--agents`) Claude Code and Codex against a local fake endpoint. |
| `vm_gate.py` | Checks on the VM before any collection: isolation and capture, DeepSeek Anthropic and Responses APIs through the gateway, key-log decryption, both agents through the gateway, and the traffic of each agent straight to DeepSeek compared with its traffic through the gateway. |
| `vm_setup.sh` | Installs the system packages (including ethtool), Node.js 22 and the pinned agent versions on Ubuntu 24.04. |

## VM gate

    git clone https://github.com/anasbiswas1/unexplained-traffic.git
    cd unexplained-traffic
    sudo bash harness/vm_setup.sh
    export DEEPSEEK_API_KEY=...        # kept in the environment only, never written to disk
    sudo -E python3 harness/vm_gate.py

Output goes to `data/harness/vm_gate/<time>/` (git-ignored): captures, call logs, tool logs, agent output, a
`net.json` per run (uplink, MTU, offload states, capture counts) and `vm_gate.json`. Key logs are deleted when the gate
ends.

## Agent settings on DeepSeek

- Claude Code, from DeepSeek's Claude Code guide: `ANTHROPIC_BASE_URL` (the gateway, which forwards to
  `https://api.deepseek.com/anthropic`), `ANTHROPIC_AUTH_TOKEN`, `ANTHROPIC_MODEL=deepseek-flash[1m]`,
  `ANTHROPIC_DEFAULT_OPUS_MODEL=deepseek-flash[1m]`, `ANTHROPIC_DEFAULT_SONNET_MODEL=deepseek-flash[1m]`,
  `ANTHROPIC_DEFAULT_HAIKU_MODEL=deepseek-flash`. Local MCP tools reach the model as ordinary tool definitions.
- Codex, from DeepSeek's Codex guide: provider `base_url` (the gateway, which forwards to `https://api.deepseek.com`),
  `wire_api = "responses"`, model `deepseek-flash`; the key is read from `DEEPSEEK_API_KEY` through `env_key`.

## Fixed rules

- Benign tasks only. No real credentials and no real user data in any run.
- The decrypted view of a capture exists only to label which bytes were real content, as a check on the byte-budget
  analysis. It is never an input to the detector.
- Key logs are deleted with the testbed and never released. Captures, key logs and call logs live under `data/` and
  are never committed.

In [ ]:
import py_compile
os.chmod('harness/vm_setup.sh', 0o755)
for f_ in ('netns.py', 'vm_gate.py'):
    py_compile.compile(f'harness/{f_}', doraise=True)
for f_ in ('netns.py', 'vm_gate.py', 'vm_setup.sh', 'README.md'):
    print(f'{f_:14s} {os.path.getsize(f"harness/{f_}"):>8,d} bytes')

## Stage B. Unit checks and the offline gateway check

In [ ]:
import importlib
sys.path.insert(0, 'harness')
import netns, vm_gate
importlib.reload(netns)
importlib.reload(vm_gate)
C, E = '10.200.13.2', '1.2.3.4'
UNIT = []

# U1 to U5: upload measures and the upstream-drop rule (as in 02b)
seg = [['0', C, '1', '1000'], ['0', C, '1001', '1000'], ['0', C, '1001', '1000'], ['0', C, '2001', '500'],
       ['0', E, '1', '700'], ['1', C, '1', '1000'], ['1', C, '501', '1000'], ['2', C, '', '']]
rec = [['0', C, '1495,995'], ['1', C, '1495'], ['0', E, '695']]
m = vm_gate.upload_measures(seg, rec, C)
UNIT.append(('U1 upload with a full resend', (m['0']['up_raw'], m['0']['up_tcp'], m['0']['resent'], m['0']['up_tls'],
                                              m['0']['down_raw']) == (3500, 2500, 1000, 2500, 700), m['0']))
UNIT.append(('U2 upload with a partial resend', (m['1']['up_raw'], m['1']['up_tcp'], m['1']['resent'],
                                                  m['1']['up_tls']) == (2000, 1500, 500, 1500), m['1']))
rows = [['99.0', E, '443', '40000', 'True', 'False'],
        ['100.52', E, '443', '40000', 'True', 'False'], ['100.53', C, '40000', '443', '1', '0'],
        ['200.10', C, '40001', '443', 'True', 'False'], ['200.11', E, '443', '40001', 'False', 'True'],
        ['300.20', E, '443', '40002', '0', '1'], ['400.20', E, '443', '40003', '1', '0']]
ev = vm_gate.close_rows_to_events(rows, C)
base = dict(seq=1, status=200, path='/v1/messages?beta=true')
calls = [dict(base, conn=1, t=100.0, t_resp_head=100.5, resp_body_bytes=0, up_local=f'{C}:40000',
              error='IncompleteReadError: 0 bytes read', error_side='upstream'),
         dict(base, conn=2, t=200.0, t_resp_head=200.05, resp_body_bytes=0, up_local=f'{C}:40001',
              error='ConnectionResetError: x', error_side='upstream'),
         dict(base, conn=3, t=300.0, t_resp_head=300.1, resp_body_bytes=900, up_local=f'{C}:40002', error=None,
              error_side=None),
         dict(base, conn=4, t=300.0, t_resp_head=300.1, resp_body_bytes=900, up_local=f'{C}:40002',
              error='IncompleteReadError: y', error_side='upstream'),
         dict(base, conn=5, t=400.0, t_resp_head=400.1, resp_body_bytes=84, up_local=f'{C}:40003',
              error='ConnectionResetError: Connection lost', error_side='agent')]
d = vm_gate.upstream_drops(calls, ev)
UNIT.append(('U3 endpoint drop after the head, before any body',
             len(d) == 2 and d[0]['conn'] == 1 and d[0]['phase'] == 'after the 200 response head, before any body'
             and d[0]['closed_by'] == 'endpoint FIN' and d[0]['seconds_after_head'] == 0.02, d[:1]))
UNIT.append(('U4 endpoint drop inside the body', len(d) == 2 and d[1]['conn'] == 4
             and d[1]['phase'] == 'inside the body, after 900 bytes' and d[1]['closed_by'] == 'endpoint RST', d[1:]))
UNIT.append(('U5 client close, agent close and clean calls are not drops',
             {x['conn'] for x in d}.isdisjoint({2, 3, 5}), [x['conn'] for x in d]))

# U6: download resends come from segments tshark marks as resent, endpoint side only
m6 = vm_gate.upload_measures(seg, rec, C, flagged=[['0', E, '1408'], ['0', C, '1000'], ['1', E, '700']])
UNIT.append(('U6 download resends', (m6['0']['down_resent'], m6['1']['down_resent'], m6['0']['resent']) ==
             (1408, 700, 1000), {k: v['down_resent'] for k, v in m6.items()}))

# U7: uplink, route mtu attribute and link MTU parsing
gce = "default via 10.128.0.1 dev ens4 proto dhcp src 10.128.0.2 metric 100"
link = ("2: ens4: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1460 qdisc mq state UP mode DEFAULT group default qlen 1000"
        "\\    link/ether 42:01:0a:80:00:02 brd ff:ff:ff:ff:ff:ff")
r7 = (netns.parse_default_route(gce), netns.parse_default_route("default via 10.0.0.1 dev eth0 mtu 1400"),
      netns.parse_default_route("default via 10.0.0.1 dev eth0 mtu lock 1380 metric 5"), netns.parse_link_mtu(link))
UNIT.append(('U7 uplink and MTU parsing', r7 == (('ens4', None), ('eth0', 1400), ('eth0', 1380), 1460), r7))

# U8: offload states
ON = ("Features for utn13:\nrx-checksumming: on\ntx-checksumming: on\nscatter-gather: on\n"
      "tcp-segmentation-offload: on\n\ttx-tcp-segmentation: on\ngeneric-segmentation-offload: on\n"
      "generic-receive-offload: off\nlarge-receive-offload: off [fixed]\n")
OFF = ON.replace("tcp-segmentation-offload: on", "tcp-segmentation-offload: off").replace(
    "generic-segmentation-offload: on", "generic-segmentation-offload: off")
r8 = (netns.offloads_off(netns.parse_offloads(ON)), netns.offloads_off(netns.parse_offloads(OFF)),
      netns.offloads_off(netns.parse_offloads(OFF.replace("generic-receive-offload: off\n", ""))))
UNIT.append(('U8 offload states', r8 == (False, True, False), r8))

# U9: tcpdump closing counts
done = "tcpdump: listening on utn13\n1875 packets captured\n1875 packets received by filter\n0 packets dropped by kernel"
tail = "1616 packets captured\n1737 packets received by filter\n0 packets dropped by kernel"
drop = "900 packets captured\n950 packets received by filter\n50 packets dropped by kernel"
r9 = [netns.parse_capture_stats(t)['complete'] for t in (done, tail, drop, "")]
UNIT.append(('U9 capture counts', r9 == [True, False, False, False], r9))

# U10: wire verdict on a clean capture and on each fault
offs = {'uth13': netns.parse_offloads(OFF), 'utn13': netns.parse_offloads(OFF)}
clean = dict(mtu=1460, uplink='ens4', uplink_mtu=1460, host_if_mtu=1460, ns_if_mtu=1460, offloads=offs,
             icmp_frag_needed=0, icmp_next_hop_mtu=[], packets=900, largest_packet=1460, up_resent=0,
             down_resent=1408, capture=dict(captured=900, received=900, dropped_kernel=0, dropped_interface=None,
                                            complete=True))
faults = dict(mtu_mismatch=dict(host_if_mtu=1500, ns_if_mtu=1500),
              offloads_on=dict(offloads={'utn13': netns.parse_offloads(ON)}),
              icmp=dict(icmp_frag_needed=2, icmp_next_hop_mtu=['1460']),
              block=dict(largest_packet=65212),
              resent=dict(up_resent=12852),
              tail=dict(capture=dict(captured=1616, received=1737, dropped_kernel=0, dropped_interface=None,
                                     complete=False)))
v_clean = vm_gate.wire_verdict(clean)
v_edge = vm_gate.wire_verdict(dict(clean, up_resent=1420))
v_faults = {k: vm_gate.wire_verdict(dict(clean, **f)) for k, f in faults.items()}
UNIT.append(('U10 wire verdict', v_clean[0] and v_edge[0] and not any(v[0] for v in v_faults.values())
             and all(len(v[1]) == 1 for v in v_faults.values()),
             dict(clean=v_clean, at_tolerance=v_edge[0], faults={k: v[1] for k, v in v_faults.items()})))
for name, ok, value in UNIT:
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {value}")
UNIT_PASS = all(ok for _, ok, _ in UNIT)

In [ ]:
import json, subprocess
import pandas as pd
import reslog
OUT = 'data/harness/offline_02c'
r = subprocess.run([sys.executable, 'harness/offline_check.py', '--out', OUT], capture_output=True, text=True,
                   timeout=900)
print(r.stdout[-8000:]); print(r.stderr[-3000:])
with open(f'{OUT}/offline_check.json') as f:
    RES = json.load(f)
RES += [dict(check=name, where='offline', passed=bool(ok), detail=str(value)) for name, ok, value in UNIT]
TAB = pd.DataFrame(RES)[['check', 'where', 'passed', 'detail']]
TAB.to_json('reports/harness/offline_check_02c.json', orient='records', indent=1)
DECISION = 'PASS' if (TAB.passed == True).all() else 'FAIL'
mark = lambda p: 'PASS' if p is True else ('SKIP' if p is None else 'FAIL')
TEXT = (f'DECISION: {DECISION} (offline checks O1-O13 and unit checks U1-U10, run in Colab; the veth, capture and '
        f'wire checks themselves run on the VM in V2 and V8)\n\n'
        + '\n'.join(f'{mark(p)}  {c}: {d}' for c, p, d in zip(TAB.check, TAB.passed, TAB.detail)))
with open('reports/harness/decision_02c.txt', 'w') as f:
    f.write(TEXT + '\n')
reslog.log_result('02c capture fidelity: offline and unit checks', TEXT)
pd.set_option('display.max_colwidth', 400)
display(TAB)
assert DECISION == 'PASS', 'offline or unit checks failed: see the table'

## Stage C. Commit

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", 'nb02c: capture fidelity before collection: netns.py sets both veth ends to the uplink MTU (default-route interface, or the route mtu when smaller; checked and recorded), turns TSO/GSO/GRO off on both ends (checked and recorded), runs tcpdump in immediate mode and records its closing counts; vm_gate.py writes net.json per run and fails a run in V2 and V8 on a veth MTU or offload mismatch, ICMP fragmentation-needed, a captured packet above the MTU, upload resends above one full-size segment, or an incomplete capture (download resends reported); V0 reports the uplink MTU and needs ethtool; vm_setup.sh installs ethtool; unit checks U1-U10; reports/harness/offline_check_02c.json, decision_02c.txt'], capture_output=True, text=True)
print(r.stdout); print(r.stderr)